In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
!pip install jiwer editdistance evaluate

In [13]:
import os
import re
import itertools
import numpy as np
import pandas as pd
import torch
import evaluate
import editdistance
from datasets import Dataset, Audio
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from jiwer import wer

In [14]:
file_path = "/content/drive/MyDrive/cleaned_ag/ag_data.xlsx"
df = pd.read_excel(file_path)

# إعادة تسمية الأعمدة لتتوافق مع الكود
df = df.rename(columns={"audio_path": "path", "Northern_aghwarDialect": "text"})

# تحويل الـ DataFrame إلى Dataset الخاص بـ Hugging Face
dataset = Dataset.from_pandas(df)
dataset = dataset.cast_column("path", Audio(sampling_rate=16000))


In [15]:
file_path = "/content/drive/MyDrive/cleaned_amman/Amman_data.xlsx"
df = pd.read_excel(file_path)

# إعادة تسمية الأعمدة لتتوافق مع الكود
df = df.rename(columns={"audio_path": "path", "AmmanDialect": "text"})

# تحويل الـ DataFrame إلى Dataset الخاص بـ Hugging Face
dataset = Dataset.from_pandas(df)
dataset = dataset.cast_column("path", Audio(sampling_rate=16000))


In [16]:
@dataclass
class DataCollatorWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # حشو الميزات الصوتية
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        if batch["input_features"].ndim == 4:
            batch["input_features"] = batch["input_features"].squeeze(1)

        # حشو النصوص (Labels)
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # استبدال الـ padding بـ -100 لحساب الـ loss بشكل صحيح وإهمال الحشو
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels

        return batch

data_collator = DataCollatorWithPadding(processor=processor)

In [17]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # استبدال الـ -100 بـ pad_token_id لكي لا تحدث أخطاء أثناء التفكيك
    labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)

    # تحويل الـ IDs إلى نصوص عربية حقيقية
    pred_str = processor.tokenizer.batch_decode(predictions, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(labels, skip_special_tokens=True)

    # حساب المقاييس
    wer_score = wer_metric.compute(predictions=pred_str, references=label_str)
    cer_score = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer_score * 100, "cer": cer_score * 100}

### **ag**

In [18]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import torch
import pandas as pd
import soundfile as sf
import os

model_path = "/content/drive/MyDrive/whisper_large_BEST_irbid"

model = WhisperForConditionalGeneration.from_pretrained(
    model_path,
    use_safetensors=True
)

processor = WhisperProcessor.from_pretrained(model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

file_path = "/content/drive/MyDrive/cleaned_ag/ag_data.xlsx"
df = pd.read_excel(file_path)

def fix_audio_path(old_path):
    if not isinstance(old_path, str):
        return old_path

    filename = os.path.basename(old_path)

    new_base_path = "/content/drive/MyDrive/cleaned_ag/audio/"
    return os.path.join(new_base_path, filename)

df["audio_path"] = df["audio_path"].apply(fix_audio_path)

results = []

for idx, row in df.iterrows():

    audio_file = row["audio_path"]
    reference_text = row["Northern_aghwarDialect"]

    try:
        audio_array, sr = sf.read(audio_file)

        inputs = processor(
            audio_array,
            sampling_rate=16000,
            return_tensors="pt"
        ).input_features.to(device)

        with torch.no_grad():
            predicted_ids = model.generate(inputs)

        transcription = processor.batch_decode(
            predicted_ids,
            skip_special_tokens=True
        )[0]

        results.append({
            "audio_file": audio_file,
            "reference": reference_text,
            "prediction": transcription
        })

        print(f"Processed {idx+1}/{len(df)}")

    except Exception as e:
        print(f"Error in {audio_file}: {e}")

results_df = pd.DataFrame(results)

results_df.to_csv(
    "whisper_ag_finetuned_irbid_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ Saved:whisper_ag_finetuned_irbid_results.csv")

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Processed 1/1500
Processed 2/1500
Processed 3/1500
Processed 4/1500
Processed 5/1500
Processed 6/1500
Processed 7/1500
Processed 8/1500
Processed 9/1500
Processed 10/1500
Processed 11/1500
Processed 12/1500
Processed 13/1500
Processed 14/1500
Processed 15/1500
Processed 16/1500
Processed 17/1500
Processed 18/1500
Processed 19/1500
Processed 20/1500
Processed 21/1500
Processed 22/1500
Processed 23/1500
Processed 24/1500
Processed 25/1500
Processed 26/1500
Processed 27/1500
Processed 28/1500
Processed 29/1500
Processed 30/1500
Processed 31/1500
Processed 32/1500
Processed 33/1500
Processed 34/1500
Processed 35/1500
Processed 36/1500
Processed 37/1500
Processed 38/1500
Processed 39/1500
Processed 40/1500
Processed 41/1500
Processed 42/1500
Processed 43/1500
Processed 44/1500
Processed 45/1500
Processed 46/1500
Processed 47/1500
Processed 48/1500
Processed 49/1500
Processed 50/1500
Processed 51/1500
Processed 52/1500
Processed 53/1500
Processed 54/1500
Processed 55/1500
Processed 56/1500
P

In [19]:

results_df = pd.read_csv("whisper_ag_finetuned_irbid_results.csv")

references = results_df["reference"].tolist()
predictions = results_df["prediction"].tolist()


wer_score = wer(references, predictions)


total_cer = sum(editdistance.eval(r, p) for r, p in zip(references, predictions))
total_chars = sum(len(r) for r in references)
cer_score = total_cer / total_chars

print(f"📊 Word Error Rate (WER): {wer_score:.4f}")
print(f"📊 Character Error Rate (CER): {cer_score:.4f}")

📊 Word Error Rate (WER): 0.1638
📊 Character Error Rate (CER): 0.0560


In [20]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import torch
import pandas as pd
import soundfile as sf
import os

# 1️⃣ تحميل الموديل المدرب تبعك
model_path = "/content/drive/MyDrive/whisper_large_BEST_irbid"

model = WhisperForConditionalGeneration.from_pretrained(
    model_path,
    use_safetensors=True
)

processor = WhisperProcessor.from_pretrained(model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# 2️⃣ قراءة البيانات من ملف Excel
file_path = "/content/drive/MyDrive/cleaned_amman/Amman_data.xlsx"
df = pd.read_excel(file_path)

def fix_audio_path(old_path):
    if not isinstance(old_path, str):
        return old_path

    filename = os.path.basename(old_path)

    new_base_path = "/content/drive/MyDrive/cleaned_amman/audio/"
    return os.path.join(new_base_path, filename)

df["audio_path"] = df["audio_path"].apply(fix_audio_path)

results = []

for idx, row in df.iterrows():

    audio_file = row["audio_path"]
    reference_text = row["AmmanDialect"]

    try:
        audio_array, sr = sf.read(audio_file)

        inputs = processor(
            audio_array,
            sampling_rate=16000,
            return_tensors="pt"
        ).input_features.to(device)

        with torch.no_grad():
            predicted_ids = model.generate(inputs)

        transcription = processor.batch_decode(
            predicted_ids,
            skip_special_tokens=True
        )[0]

        results.append({
            "audio_file": audio_file,
            "reference": reference_text,
            "prediction": transcription
        })

        print(f"Processed {idx+1}/{len(df)}")

    except Exception as e:
        print(f"Error in {audio_file}: {e}")

# 4️⃣ حفظ النتائج
results_df = pd.DataFrame(results)

results_df.to_csv(
    "whisperag_finetuned_amman_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ Saved:whisperag_finetuned_amman_results.csv")

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Processed 1/1500
Processed 2/1500
Processed 3/1500
Processed 4/1500
Processed 5/1500
Processed 6/1500
Processed 7/1500
Processed 8/1500
Processed 9/1500
Processed 10/1500
Processed 11/1500
Processed 12/1500
Processed 13/1500
Processed 14/1500
Processed 15/1500
Processed 16/1500
Processed 17/1500
Processed 18/1500
Processed 19/1500
Processed 20/1500
Processed 21/1500
Processed 22/1500
Processed 23/1500
Processed 24/1500
Processed 25/1500
Processed 26/1500
Processed 27/1500
Processed 28/1500
Processed 29/1500
Processed 30/1500
Processed 31/1500
Processed 32/1500
Processed 33/1500
Processed 34/1500
Processed 35/1500
Processed 36/1500
Processed 37/1500
Processed 38/1500
Processed 39/1500
Processed 40/1500
Processed 41/1500
Processed 42/1500
Processed 43/1500
Processed 44/1500
Processed 45/1500
Processed 46/1500
Processed 47/1500
Processed 48/1500
Processed 49/1500
Processed 50/1500
Processed 51/1500
Processed 52/1500
Processed 53/1500
Processed 54/1500
Processed 55/1500
Processed 56/1500
P

In [21]:

results_df = pd.read_csv("whisperag_finetuned_amman_results.csv")

references = results_df["reference"].tolist()
predictions = results_df["prediction"].tolist()


wer_score = wer(references, predictions)


total_cer = sum(editdistance.eval(r, p) for r, p in zip(references, predictions))
total_chars = sum(len(r) for r in references)
cer_score = total_cer / total_chars

print(f"📊 Word Error Rate (WER): {wer_score:.4f}")
print(f"📊 Character Error Rate (CER): {cer_score:.4f}")

📊 Word Error Rate (WER): 0.1861
📊 Character Error Rate (CER): 0.0548
